# 13 Aligned Transformer + XGBoost Ablation

This notebook runs structured-feature ablations for the separate aligned Transformer + XGBoost hybrid model. It does not modify notebook 09 outputs.

## Ablation scope

- run `vitals_only`, `vitals_labs`, and `structured_full` variants
- keep the original ablation pipeline untouched
- save all hybrid ablation outputs into a separate processed stage
- warning: this notebook retrains the hybrid model for each selected horizon and variant

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT


PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn xgboost


In [3]:
import pandas as pd

from src.evaluation.aligned_transformer_xgboost_ablation import (
    build_aligned_transformer_xgboost_ablation_plan,
    run_aligned_transformer_xgboost_ablation_suite,
)
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest


In [4]:
{
    'aligned_transformer_xgboost': config['aligned_transformer_xgboost'],
    'aligned_transformer_xgboost_ablation': config['aligned_transformer_xgboost_ablation'],
}


{'aligned_transformer_xgboost': {'output_stage': '10_aligned_transformer_xgboost',
  'structured_encoder': 'transformer',
  'text_encoder_mode': 'frozen_embedding',
  'hidden_dim': 128,
  'aligned_dim': 256,
  'dropout': 0.2,
  'batch_size': 8,
  'epochs': 10,
  'learning_rate': 0.0003,
  'weight_decay': 0.0001,
  'gradient_clip_norm': 1.0,
  'scheduler': 'cosine',
  'min_learning_rate': 5e-05,
  'early_stopping_patience': 4,
  'threshold_selection_metric': 'f1',
  'max_structured_steps': 48,
  'max_text_windows': 8,
  'max_tokens_per_window': 128,
  'text_embedding_dim': 768,
  'structured_aggregations': ['mean', 'min', 'max', 'last'],
  'include_missingness': True,
  'include_static_categoricals': True,
  'include_note_metadata': True,
  'include_clinical_event_features': True,
  'clinical_event_lookback_hours': 48,
  'xgboost_model': 'xgboost'},
 'aligned_transformer_xgboost_ablation': {'output_stage': '13_aligned_transformer_xgboost_ablation',
  'executable_variants': ['vitals_only

## Select horizons to ablate

In [5]:
selected_horizons = [f"horizon_{int(h)}h" for h in config.get('aligned_transformer_xgboost_ablation', {}).get('default_horizons', [24])]
selected_horizons


['horizon_6h', 'horizon_12h', 'horizon_24h']

## Load selected structured horizon tables

In [6]:
horizon_tables = {}
for dataset_name in selected_horizons:
    path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    assert path.exists(), f'Missing structured horizon dataset: {dataset_name}. Run 04_feature_engineering first.'
    horizon_tables[dataset_name] = pd.read_csv(
        path,
        parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'],
        low_memory=False,
    )

{key: value.shape for key, value in horizon_tables.items()}


{'horizon_6h': (1268294, 90),
 'horizon_12h': (1131670, 90),
 'horizon_24h': (860480, 90)}

## Run hybrid ablation suite

In [7]:
ablation_output_dir = paths['processed_data_dir'] / config.get('aligned_transformer_xgboost_ablation', {}).get('output_stage', '13_aligned_transformer_xgboost_ablation')

artifacts = run_aligned_transformer_xgboost_ablation_suite(
    horizon_tables=horizon_tables,
    config=config,
    processed_dir=paths['processed_data_dir'],
    extracted_dir=paths['extracted_data_dir'],
    output_dir=ablation_output_dir,
    device=config.get('aligned_transformer_xgboost_ablation', {}).get('device', 'auto'),
)

planned_matrix_df = artifacts['aligned_transformer_xgboost_ablation_plan']
hybrid_ablation_results_df = artifacts['aligned_transformer_xgboost_ablation_results']
encoder_ablation_results_df = artifacts['aligned_transformer_encoder_ablation_results']

display(planned_matrix_df)
display(hybrid_ablation_results_df)
display(encoder_ablation_results_df)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,variant_name,description,implemented_now
0,vitals_only,Structured vitals subset only,True
1,vitals_labs,Vitals plus laboratory subset,True
2,structured_full,All structured EHR features,True


,dataset_name,split,model_name,structured_encoder,text_input_mode,text_embedding_backend,decision_threshold,n_features,n_aligned_features,n_structured_features,...,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,variant_name
0,horizon_12h,test,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.175241,707,256,428,...,0.958159,0.929006,0.005310,0.995272,0.958159,229,25,5263,10,structured_full
1,horizon_12h,val,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.175241,707,256,428,...,0.955665,0.932692,0.004690,0.996366,0.955665,194,19,5209,9,structured_full
2,horizon_12h,test,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.134501,630,256,360,...,0.309623,0.390501,0.034149,0.987519,0.309623,74,66,5222,165,vitals_labs
3,horizon_12h,val,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.134501,630,256,360,...,0.280788,0.356250,0.030553,0.988523,0.280788,57,60,5168,146,vitals_labs
4,horizon_12h,test,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.090379,430,256,160,...,0.343096,0.296029,0.037219,0.955938,0.343096,82,233,5055,157,vitals_only
5,horizon_12h,val,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.090379,430,256,160,...,0.275862,0.230453,0.034072,0.956580,0.275862,56,227,5001,147,vitals_only
6,horizon_24h,test,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.319207,707,256,428,...,0.971429,0.949721,0.003340,0.996617,0.971429,170,13,3830,5,structured_full
7,horizon_24h,val,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.319207,707,256,428,...,0.958904,0.924092,0.005494,0.995586,0.958904,140,17,3834,6,structured_full
8,horizon_24h,test,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.073288,630,256,360,...,0.354286,0.341598,0.035591,0.967213,0.354286,62,126,3717,113,vitals_labs
9,horizon_24h,val,aligned_transformer_xgboost,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,0.073288,630,256,360,...,0.294521,0.287625,0.032586,0.971436,0.294521,43,110,3741,103,vitals_labs


,dataset_name,split,model_name,structured_encoder,text_input_mode,text_embedding_backend,aligned_dim,n_examples,decision_threshold,loss,...,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,variant_name
0,horizon_12h,test,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,5527,0.027812,6.153160,...,0.276151,0.253359,0.041242,0.959153,0.276151,66,216,5072,173,structured_full
1,horizon_12h,val,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,5431,0.027812,5.318902,...,0.231527,0.199575,0.035767,0.957728,0.231527,47,221,5007,156,structured_full
2,horizon_12h,test,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,5527,0.036422,4.901782,...,0.288703,0.194366,0.041465,0.923979,0.288703,69,402,4886,170,vitals_labs
3,horizon_12h,val,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,5431,0.036422,4.264142,...,0.256158,0.149640,0.035992,0.915838,0.256158,52,440,4788,151,vitals_labs
4,horizon_12h,test,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,5527,0.037951,5.518952,...,0.326360,0.257851,0.040445,0.945537,0.326360,78,288,5000,161,vitals_only
5,horizon_12h,val,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,5431,0.037951,4.860087,...,0.211823,0.160748,0.035913,0.944721,0.211823,43,289,4939,160,vitals_only
6,horizon_24h,test,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,4018,0.062811,5.237380,...,0.171429,0.200669,0.040978,0.975540,0.171429,30,94,3749,145,structured_full
7,horizon_24h,val,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,3997,0.062811,4.534783,...,0.198630,0.219697,0.034653,0.976889,0.198630,29,89,3762,117,structured_full
8,horizon_24h,test,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,4018,0.052431,5.603794,...,0.148571,0.179310,0.041454,0.976841,0.148571,26,89,3754,149,vitals_labs
9,horizon_24h,val,aligned_transformer_encoder,transformer,frozen_embedding,transformer:emilyalsentzer/Bio_ClinicalBERT,256,3997,0.052431,4.936424,...,0.157534,0.178988,0.035252,0.977149,0.157534,23,88,3763,123,vitals_labs


## Build summary views

In [8]:
summary_columns = [
    'dataset_name',
    'variant_name',
    'split',
    'model_name',
    'auroc',
    'auprc',
    'accuracy',
    'precision',
    'recall',
    'f1',
    'decision_threshold',
]

hybrid_summary_df = (
    hybrid_ablation_results_df.loc[:, [column for column in summary_columns if column in hybrid_ablation_results_df.columns]]
    .sort_values(['dataset_name', 'variant_name', 'split'])
    .reset_index(drop=True)
    if not hybrid_ablation_results_df.empty else pd.DataFrame()
)

hybrid_pivot_df = (
    hybrid_ablation_results_df.pivot_table(
        index=['dataset_name', 'variant_name'],
        columns='split',
        values=['auroc', 'auprc', 'f1'],
        aggfunc='first',
    )
    if not hybrid_ablation_results_df.empty else pd.DataFrame()
)

display(hybrid_summary_df)
display(hybrid_pivot_df)


,dataset_name,variant_name,split,model_name,auroc,auprc,accuracy,precision,recall,f1,decision_threshold
0,horizon_12h,structured_full,test,aligned_transformer_xgboost,0.998574,0.960815,0.993667,0.901575,0.958159,0.929006,0.175241
1,horizon_12h,structured_full,val,aligned_transformer_xgboost,0.998807,0.965827,0.994844,0.910798,0.955665,0.932692,0.175241
2,horizon_12h,vitals_labs,test,aligned_transformer_xgboost,0.816277,0.373448,0.958205,0.528571,0.309623,0.390501,0.134501
3,horizon_12h,vitals_labs,val,aligned_transformer_xgboost,0.797245,0.334071,0.962070,0.487179,0.280788,0.356250,0.134501
4,horizon_12h,vitals_only,test,aligned_transformer_xgboost,0.776987,0.246888,0.929437,0.260317,0.343096,0.296029,0.090379
5,horizon_12h,vitals_only,val,aligned_transformer_xgboost,0.715652,0.163436,0.931136,0.197880,0.275862,0.230453,0.090379
6,horizon_24h,structured_full,test,aligned_transformer_xgboost,0.999236,0.978877,0.995520,0.928962,0.971429,0.949721,0.319207
7,horizon_24h,structured_full,val,aligned_transformer_xgboost,0.998229,0.952739,0.994246,0.891720,0.958904,0.924092,0.319207
8,horizon_24h,vitals_labs,test,aligned_transformer_xgboost,0.802235,0.327766,0.940518,0.329787,0.354286,0.341598,0.073288
9,horizon_24h,vitals_labs,val,aligned_transformer_xgboost,0.764404,0.206915,0.946710,0.281046,0.294521,0.287625,0.073288


auprc               auroc            \
split                             test       val      test       val   
dataset_name variant_name                                              
horizon_12h  structured_full  0.960815  0.965827  0.998574  0.998807   
             vitals_labs      0.373448  0.334071  0.816277  0.797245   
             vitals_only      0.246888  0.163436  0.776987  0.715652   
horizon_24h  structured_full  0.978877  0.952739  0.999236  0.998229   
             vitals_labs      0.327766  0.206915  0.802235  0.764404   
             vitals_only      0.154293  0.082710  0.725021  0.679909   
horizon_6h   structured_full  0.974953  0.977741  0.998627  0.998984   
             vitals_labs      0.367220  0.397197  0.828491  0.839447   
             vitals_only      0.245533  0.310660  0.801777  0.802104   

                                    f1            
split                             test       val  
dataset_name variant_name                         
horizon_12h  structured_full  0.929006  0.932692  
             vitals_labs      0.390501  0.356250  
             vitals_only      0.296029  0.230453  
horizon_24h  structured_full  0.949721  0.924092  
             vitals_labs      0.341598  0.287625  
             vitals_only      0.205426  0.156448  
horizon_6h   structured_full  0.954248  0.943005  
             vitals_labs      0.408526  0.448669  
             vitals_only      0.317518  0.351449

## Save ablation artifacts

In [9]:
saved_paths = save_dataframe_bundle(artifacts, ablation_output_dir)
manifest_path = paths['manifests_dir'] / '13_aligned_transformer_xgboost_ablation_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='13_aligned_transformer_xgboost_ablation',
    config=config,
    extra={
        'selected_horizons': selected_horizons,
        'saved_artifacts': saved_paths,
    },
)

saved_paths, manifest_path


({'horizon_6h_vitals_only_aligned_transformer_encoder_val_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/13_aligned_transformer_xgboost_ablation/horizon_6h_vitals_only_aligned_transformer_encoder_val_predictions.csv',
  'horizon_6h_vitals_only_aligned_transformer_encoder_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/13_aligned_transformer_xgboost_ablation/horizon_6h_vitals_only_aligned_transformer_encoder_test_predictions.csv',
  'horizon_6h_vitals_only_aligned_transformer_xgboost_val_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/13_aligned_transformer_xgboost_ablation/horizon_6h_vitals_only_aligned_transformer_xgboost_val_predictions.csv',
  'horizon_6h_vitals_only_aligned_transformer_xgboost_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/13_aligned_transformer_xgboost_ablation/horizon_6h_vitals_only